# Ex1 - Filtering and Sorting Data



### Step 1. Import the necessary libraries

In [2]:
import numpy as np
import pandas as pd
import csv

### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv) and assign it to a variable called chipo.

In [3]:
chipo= pd.read_csv("chipotle.tsv",sep='\t')#sep="\t" para indicar que los campos están separados por una tabulacion.
chipo

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


### Step 3. Name of the max valued product

In [4]:
'''Tenemos varios items que han pedidio varias veces, esto va a hacer que el precio que aparezca sea por pedido y no el del articulo particular'''
items_rep = chipo["quantity"].unique()
items_rep


array([ 1,  2,  3,  4,  5, 15,  7,  8, 10])

In [5]:
nueva_lista = chipo.copy()
nueva_lista["item_price"] = nueva_lista["item_price"].replace("[\\$]","",regex=True)#esto ultimo me elimina los espacios en blaco al principio y al fianal
#elimino de la lista de precios el simbolo del dolar y los espacios para poder convertilo a float y poder operar
nueva_lista["item_price"] = nueva_lista["item_price"].astype(float)
#he puesto dos \\ para que me reconoca el caracter $ ya que \ es un escape de python y lo interpreta asi

In [6]:
#para obtener los precios reales basta con dividir cada precio por su cantidad
#si es uno no cambia y si es otro valor me dara el oprecio por unidad
nueva_lista["unit_price"] = nueva_lista["item_price"] / nueva_lista["quantity"]
print(nueva_lista["unit_price"].max())
print(nueva_lista["unit_price"].idxmax())
print(nueva_lista["item_name"][607])


11.89
281
Steak Salad Bowl


### Step 4. How many products cost more than $10.00?

In [7]:

precios_mayores_10 = nueva_lista[nueva_lista["item_price"]>9.99]
precios_mayores_10

,order_id,quantity,item_name,choice_description,item_price,unit_price
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98,8.49
5,3,1,Chicken Bowl,"[Fresh Tomato Salsa (Mild), [Rice, Cheese, Sou...",10.98,10.98
7,4,1,Steak Burrito,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",11.75,11.75
13,7,1,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",11.25,11.25
23,12,1,Chicken Burrito,"[[Tomatillo-Green Chili Salsa (Medium), Tomati...",10.98,10.98
...,...,...,...,...,...,...
4610,1830,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",11.75,11.75
4611,1830,1,Veggie Burrito,"[Tomatillo Green Chili Salsa, [Rice, Fajita Ve...",11.25,11.25
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",11.75,11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",11.75,11.75


In [8]:
productos_unicos= precios_mayores_10["item_name"].unique()
print(f"Hay {len(productos_unicos)} productos que cuestas 10 o más $")


Hay 31 productos que cuestas 10 o más $


### Step 4.1: Y cuántos pedidos se han hecho con un producto de más de 10$? Es lo mismo?

In [9]:
#Tengo que ver en que pedidos tengo un producto de precio_mayor_10

In [10]:
#ya habiamos hecho una modificadion de la tabla donde teniamos los productos que valian 10 0 mas
#en esa tabla teniamos la orden de pedido, como siguen saliendo algunos repetidos hacemos un unique
orden_producto_10 =precios_mayores_10["order_id"].nunique()
#basta con contar cuantos pedidos(sin repetir) tienen un producto caro
#print(f"En {len(orden_producto_10)} pedidos encontramos un producto que vale 10 o mas dolares")
orden_producto_10

863

### Step 4.2: Y cuántos pedidos se han hecho de más de 10$? Es lo mismo?

In [11]:
#tengo que sumar en cada pedido los precios totales de los productos(con las cantidades reales) y con una mask veo si el toal supera los 10 dolares

lista1=chipo.copy()
lista1["item_price"] = lista1["item_price"].replace("[\\$]"," ",regex=True).str.strip()#esto ultimo me elimina los espacios en blaco al principio y al fianal
#elimino de la lista de precios el simbolo del dolar y los espacios para poder convertilo a float y poder operar
lista1["item_price"] = lista1["item_price"].astype(float)
#me interesan la columna de order_id(la quiero en una) y la de item_price (sumarla en cada orden)
lista1[["item_price","order_id"]]

,item_price,order_id
0,2.39,1
1,3.39,1
2,3.39,1
3,2.39,1
4,16.98,2
...,...,...
4617,11.75,1833
4618,11.75,1833
4619,11.25,1834
4620,8.75,1834


In [12]:
lista1.groupby("order_id")["item_price"].sum()
#agrupame la orden de pedido y sumame los precios

order_id
1       11.56
2       16.98
3       12.67
4       21.00
5       13.70
        ...  
1830    23.00
1831    12.90
1832    13.20
1833    23.50
1834    28.75
Name: item_price, Length: 1834, dtype: float64

In [13]:
total_precio_pedido =lista1.pivot_table(columns="order_id",values="item_price",aggfunc = sum)
total_precio_pedido
#como la orden se repetia por cada producto que añadiamos, hemos hecho una columna con cada orden y la hemos llenado con el precio total (sum) de la suma de todos los productos asociados a esa orden


C:\Users\UX490U\AppData\Local\Temp\ipykernel_16192\2319054806.py:1: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  total_precio_pedido =lista1.pivot_table(columns="order_id",values="item_price",aggfunc = sum)


order_id,1,2,3,4,5,6,7,8,9,10,...,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834
item_price,11.56,16.98,12.67,21.0,13.7,17.5,15.7,10.88,10.67,13.2,...,66.5,15.95,32.95,14.45,24.25,23.0,12.9,13.2,23.5,28.75


In [14]:
#basta ver que columna tiene un precio mayor a 10
(total_precio_pedido[total_precio_pedido>9.99]).shape[1]

1834

In [15]:
'''Segun el resultado que hemos obtenido con hacienco un mask al total de los pedidos vemos que todos los pedidos suman un coste mayor o igual a 10 dolares
    Respecto a la pregunta de si es lo mismo que el anterior apartado la respuesta es no, ya que en este caso estamos sumando el valor total
     de todos los productos pedidos en cada orden y comprobado cuando el TOTAL supera los 10 $
      En el caso anterior hemos hecho un estudio por separado de unicamente los PRODUCTOS y hemos estudiado cuales de ellos tienen un coste mayor a 10$ '''

'Segun el resultado que hemos obtenido con hacienco un mask al total de los pedidos vemos que todos los pedidos suman un coste mayor o igual a 10 dolares\n    Respecto a la pregunta de si es lo mismo que el anterior apartado la respuesta es no, ya que en este caso estamos sumando el valor total\n     de todos los productos pedidos en cada orden y comprobado cuando el TOTAL supera los 10 $\n      En el caso anterior hemos hecho un estudio por separado de unicamente los PRODUCTOS y hemos estudiado cuales de ellos tienen un coste mayor a 10$ '

### Step 4.3: Y en cuántos pedidos se ha pagado más de 10$ por un mismo producto? Es lo mismo?

In [16]:
#en este caso tenemos dos factores a considerar:
#   1. los productos que valen mas de 10$ (ya estudiados)
#   2. los productos que por la cantidad pedida tienen un coste superior a 10$
'''Necesitariamos la columna de order_id, imen_price
    para ello podemos hacer un procedimiento similar al anterior pero sin realizar la suma total de los productos
    Haremos una nueva tabla y una mask conde si en esa tabla encuentra valores >9.99 considere ese pedido'''

'Necesitariamos la columna de order_id, imen_price\n    para ello podemos hacer un procedimiento similar al anterior pero sin realizar la suma total de los productos\n    Haremos una nueva tabla y una mask conde si en esa tabla encuentra valores >9.99 considere ese pedido'

In [17]:
chipo["item_price"] = chipo["item_price"].replace("[\\$]","",regex=True).astype(float)

In [18]:

produc_ord_prec =chipo.pivot_table(index="item_name",columns="order_id",values="item_price")
#de esta forma tendriamos una tabla completa con el nombre de producto en el eje 0 y las ordenes en el eje 1 llena con los precios de los productos 
#pedidos en cada orden (Nan en los que no se pidan) y luego tendriamos que hacer una suma de cada columna(orden de pedido)
produc_ord_prec


order_id,1,2,3,4,5,6,7,8,9,10,...,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834
item_name,,,,,,,,,,,,,,,,,,,,,
6 Pack Soft Drink,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Barbacoa Bowl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Barbacoa Burrito,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.750000,NaN,9.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Barbacoa Crispy Tacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Barbacoa Salad Bowl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Barbacoa Soft Tacos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bottled Water,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.50,NaN,NaN,NaN,NaN,1.50,NaN,NaN,NaN
Bowl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Burrito,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
#FALLA EL RESULTADO
'''FALLA PQ LLEVA ASOCIADA UNA FUNCION, COMO NO LA HE DECLARADO DE SERIE LLEVA LA MEDIA
COMO TENGO EN ALGUNOS CASOS VARIOS PRODUCTOS QUE SE REPITEN CON DISTINTOS PRECIOS ME HACE LA MEDIA DE ELLOS, POR ESO DIFIERE EL RESULTADO'''
'''Tenemos que aplicar la mask ya que hemos ordenado cada orden con el precio final del producto pedido(contando las veces que se ha pedido)'''

resultado = produc_ord_prec.columns[(produc_ord_prec>10).any()]


#.columns me permite acceder a las columnas que son lo que quiero revisar (order_id)
(f"Tenemos {len(resultado)} pedidos que han pagado mas de 10$ por u mismo pedido (unico o por cantidad comprada)")

'Tenemos 827 pedidos que han pagado mas de 10$ por u mismo pedido (unico o por cantidad comprada)'

In [20]:
lista1[lista1["item_price"]>10]["order_id"].nunique()

863

In [21]:
'''al aplicar la condicion >9.99 convierto mi DF en un booleanos, ya que pandas busca en las columnas si se cumple(True) o no (False) la condicion que le paso
    con .columns le pedimos a pandas que indexe las columnas y espera un array o Serie booleqna (de 1D), pero el DF booleano (es de 2D) por eso da error
    con .any() reducimo el DF bool a una Serie bool y ya no nos da fallo en el mapeo
    '''

'al aplicar la condicion >9.99 convierto mi DF en un booleanos, ya que pandas busca en las columnas si se cumple(True) o no (False) la condicion que le paso\n    con .columns le pedimos a pandas que indexe las columnas y espera un array o Serie booleqna (de 1D), pero el DF booleano (es de 2D) por eso da error\n    con .any() reducimo el DF bool a una Serie bool y ya no nos da fallo en el mapeo\n    '

### Step 5. What is the price of each item and name it unit_price. Get only item_name and unit_price

In [22]:
#Tenemos que obtener un DF con cada producto y su precio por unidad

In [26]:
lista2= chipo.copy()
lista2["item_price"] = lista2["item_price"].replace("[\\$]"," ",regex=True).str.strip()
lista2["item_price"] = lista2["item_price"].astype(float)
#voy a crear una nueva columna de precio por unidad de producto
lista2["unit_price"]= lista2["item_price"] / lista2["quantity"]
lista2

AttributeError: Can only use .str accessor with string values!

In [20]:
len(lista2["item_name"].unique())
#para comprobar la cantidad de productos que tenemos

50

In [21]:
'''Vamos a crear una tabla donde la columna sea cada producto(asi nos evitamos las repeticiones del mismo) 
    y los valores sean los precios por unidad que hemos obtenido anteriormente'''
precio_unidad =lista2.pivot_table(columns="item_name",values="unit_price")
print("TABLA DE PRECIOS")
precio_unidad

TABLA DE PRECIOS


item_name,6 Pack Soft Drink,Barbacoa Bowl,Barbacoa Burrito,Barbacoa Crispy Tacos,Barbacoa Salad Bowl,Barbacoa Soft Tacos,Bottled Water,Bowl,Burrito,Canned Soda,...,Steak Crispy Tacos,Steak Salad,Steak Salad Bowl,Steak Soft Tacos,Veggie Bowl,Veggie Burrito,Veggie Crispy Tacos,Veggie Salad,Veggie Salad Bowl,Veggie Soft Tacos
unit_price,6.49,10.187273,9.832418,10.087273,10.64,10.0184,1.431667,7.4,7.4,1.09,...,9.952857,8.915,11.027931,9.578182,10.011882,9.602842,8.49,8.49,10.138889,9.352857


In [22]:
#DUDA: QUERIA VER SI COINCIDIAN LOS PRODUCTOS CON PRECIOS MAYORES O IGUAL A 10 DOLARES
    #no se por que no coinciden (26)
len(precio_unidad.columns[(precio_unidad>9.99).any()])

11

### Step 6. Sort by the name of the item

In [71]:

sorted_name = chipo.sort_values(["item_name","choice_description"])
print("Lista ordenada por nombre")
sorted_name
#key =lambda col: col.str.lower()) hacemos que ignore las mayusculas y minuculas para ordenar
#en este caso no hace falta porque todos los productos empiezan conn mayusculas



Lista ordenada por nombre


,order_id,quantity,item_name,choice_description,item_price
357,154,1,6 Pack Soft Drink,[Coke],6.49
743,306,1,6 Pack Soft Drink,[Coke],6.49
879,363,1,6 Pack Soft Drink,[Coke],6.49
1051,432,1,6 Pack Soft Drink,[Coke],6.49
1124,465,1,6 Pack Soft Drink,[Coke],6.49
...,...,...,...,...,...
781,322,1,Veggie Soft Tacos,"[Fresh Tomato Salsa, [Black Beans, Cheese, Sou...",8.75
1699,688,1,Veggie Soft Tacos,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",11.25
2851,1132,1,Veggie Soft Tacos,"[Roasted Chili Corn Salsa (Medium), [Black Bea...",8.49
2384,948,1,Veggie Soft Tacos,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",8.75


### Step 7. What was the quantity of the most expensive item ordered? 2 ways

In [24]:
pedido_caro=total_precio_pedido[:].max().max()
#con un solo max obtenemos todos los valores maximos por columnas, en este caso como solo hay una fila nos saca la misma tabla
#tenemos que hacer otro max para que nos saque el maximo de esos maximos
print(f"El pedido mas caro ha sido de {pedido_caro} $")

El pedido mas caro ha sido de 205.25 $


In [72]:
#cual es el item mas caro
lista2[lista2["unit_price"]== lista2["unit_price"].max()]["quantity"].sum()

np.int64(30)

In [73]:
#ahora quiero que me lo agrupe por item_name y me sume las cantidades

lista2[lista2["unit_price"]== lista2["unit_price"].max()].groupby("item_name")["quantity"].sum()

item_name
Barbacoa Salad Bowl     5
Carnitas Salad Bowl     4
Steak Salad Bowl       21
Name: quantity, dtype: int64

In [76]:
lista2[lista2["item_name"].isin(["Barbacoa Salad Bowl","Carnitas Salad Bowl","Steak Salad Bowl"])]["quantity"].sum()

#las cantidades son difefrentes ya que puede haber alguno de estos products con una version mas economica(por la descripcion del producto)


np.int64(47)

In [77]:
lista2[lista2["item_name"]== "Steak Salad Bowl"]["unit_price"].unique()
#aqui vemos qu epara un mismo producto teneos dos precios

array([11.89,  9.39])

### Step 8. How many times was a Veggie Salad Bowl ordered?

In [78]:
lista2[lista2["item_name"] == "Veggie Salad Bowl"]["quantity"].sum()#cuantos articulos

np.int64(18)

In [80]:
lista2[lista2["item_name"] == "Veggie Salad Bowl"]["order_id"].nunique()#cuantos pedidos(no tine pq coincidir)

18

In [25]:
#necesito "quantity" e item_name = veggie Salad Bowl
#haremos una tabla con las veces que se ha pedido un producto y el nombre del mismo, los valores que la llenan seran la suma de values=quantity
lista3 = chipo.copy()
cant_pro =lista3.pivot_table(columns="item_name",values="quantity",aggfunc="sum")
cant_pro


item_name,6 Pack Soft Drink,Barbacoa Bowl,Barbacoa Burrito,Barbacoa Crispy Tacos,Barbacoa Salad Bowl,Barbacoa Soft Tacos,Bottled Water,Bowl,Burrito,Canned Soda,...,Steak Crispy Tacos,Steak Salad,Steak Salad Bowl,Steak Soft Tacos,Veggie Bowl,Veggie Burrito,Veggie Crispy Tacos,Veggie Salad,Veggie Salad Bowl,Veggie Soft Tacos
quantity,55,66,91,12,10,25,211,4,6,126,...,36,4,31,56,87,97,1,6,18,8


In [26]:

VBS_suma=cant_pro["Veggie Salad Bowl"]#accedo al nombre de la columna para otener su valor
print(f"Se han pedido {VBS_suma} Veggie Salad Bowl")

Se han pedido quantity    18
Name: Veggie Salad Bowl, dtype: int64 Veggie Salad Bowl


### Step 9. How many times did someone order more than one Canned Soda?

In [81]:
lista2[(lista2["item_name"]== "Canned Soda") &(lista2["quantity"]>1)]["order_id"].nunique()
#de esta forma dejamos muchos datos fuera, en un mismo pedido puede que se hayan pedido CS y se han apuntado de uno en uno

18

In [84]:
ped_refr=lista2[(lista2["item_name"]== "Canned Soda")].groupby("order_id")["quantity"].sum()

In [85]:
len(ped_refr[ped_refr>1])

24

In [ ]:
'''NO ESTA BIEN LO SIGUIEN'''

In [27]:
#necesito las quantity>1 y el item_name = Canned Soda
x=lista3[["quantity","item_name"]]
CS_quant=x[(x["item_name"]=="Canned Soda" )& (x["quantity"]>1)]
CS_quant


,quantity,item_name
18,2,Canned Soda
51,2,Canned Soda
162,2,Canned Soda
171,2,Canned Soda
350,2,Canned Soda
352,2,Canned Soda
698,2,Canned Soda
700,2,Canned Soda
909,2,Canned Soda
1091,2,Canned Soda


In [28]:
suma =CS_quant["quantity"].sum()
print(f"Hay {suma} veces mas de una unidad de Canned Soda en los pedidos")

Hay 42 veces mas de una unidad de Canned Soda en los pedidos
